# Detectron2 segmentation for idtracker.ai — GPU stagesAnnotation happens locally in LabelMe. This notebook does the two stages thatneed a GPU: **fine-tuning** the Mask R-CNN and **exporting contours** for everyclip. The contour files come back to Drive, and idtracker.ai reads them locally.Inference decodes every frame, so **the videos themselves must be on Drive andreachable from this runtime** — not just the dataset. Section 2 verifies thatbefore anything long starts.**The export cannot finish in one Colab session.** At a plausible 10 fps, 48clips of 30 000 frames is about 40 hours. So the export is resumable: each clipwrites its own file, and rerunning skips whatever is already done. Reconnect andrun the export cell again as many times as it takes.## What to put on Drive firstRun `python tools/make_colab_bundle.py` locally, then upload to`MyDrive/idtrackerai_detectron2/`:```idtrackerai_detectron2/  colab_bundle.zip      <- from make_colab_bundle.py  dataset/              <- from labelme_to_coco.py (train.json, val.json, images/)```The **videos** can live anywhere on Drive — set `VIDEOS` in the config cell towherever they are. They do not have to sit inside the project folder, whichmatters when they are tens of GB.`model/`, `contours/` and `wheels/` are created here.Set the runtime to a GPU first: **Runtime → Change runtime type → T4/L4/A100**.

## 1. Runtime and Drive

In [ ]:
# Check the runtime is a GPU one before anything else. Everything below assumes it.
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "No GPU. Runtime -> Change runtime type -> GPU, then rerun.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ---- the only block you should need to edit --------------------------------
PROJECT  = Path('/content/drive/MyDrive/idtrackerai_detectron2')

# Where the videos actually live. Anywhere on Drive is fine — they do not have
# to be inside PROJECT. Point this at the real folder.
VIDEOS   = PROJECT / 'clips'
VIDEO_GLOB = '*.mp4'

BUNDLE   = PROJECT / 'colab_bundle.zip'
DATASET  = PROJECT / 'dataset'       # train.json, val.json, images/
MODEL    = PROJECT / 'model'         # weights land here
CONTOURS = PROJECT / 'contours'      # one .h5 per clip lands here
WHEELS   = PROJECT / 'wheels'        # cached detectron2 build

CODE     = Path('/content/code')     # bundle unpacked here
CACHE    = Path('/content/cache')    # local video copies, not on Drive

N_ANIMALS       = 5        # expected animals per frame
SCORE_THRESHOLD = 0.7
ON_OVERLAP      = 'merge'  # 'merge' keeps crossings for idtracker.ai to resolve
EPOCHS          = 40
# ----------------------------------------------------------------------------

for d in (MODEL, CONTOURS, WHEELS):
    d.mkdir(parents=True, exist_ok=True)

for label, path in [('bundle', BUNDLE), ('dataset', DATASET), ('videos', VIDEOS)]:
    print(f"{label:8s} {'OK     ' if path.exists() else 'MISSING'}  {path}")

In [ ]:
# Unpack the tools. The bundle keeps tools/ and src/ side by side because the
# exporter loads write_contours from src/ when idtracker.ai is not installed.
import shutil, zipfile

if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as zf:
    zf.extractall(CODE)

TOOLS = CODE / 'tools'
expected = [
    TOOLS / 'train_detectron2.py',
    TOOLS / 'detectron2_export_contours.py',
    TOOLS / 'check_videos.py',
    TOOLS / 'frame_preprocessing.py',
    CODE / 'src/idtrackerai/base/animals_detection/external_contours.py',
]
print('\n'.join(f"{'OK     ' if p.is_file() else 'MISSING'}  {p}" for p in expected))
if any(not p.is_file() for p in expected):
    raise SystemExit('Bundle is incomplete. Rebuild it with make_colab_bundle.py.')

## 2. Find the videosInference reads every frame of every clip, so the videos have to be on thisruntime's Drive mount. This section fails loudly now rather than forty minutesinto a run.A folder someone shared with you is **not reachable by path** until you add ashortcut to it in My Drive: open *Shared with me*, right-click the folder,*Organise → Add shortcut to Drive*. Shared drives, when enabled, appear under`/content/drive/Shareddrives/`.

In [ ]:
# Is Drive mounted, and is the video folder actually there?
import os

mount = Path('/content/drive/MyDrive')
print(f"Drive mounted:    {mount.is_dir()}")
if not mount.is_dir():
    raise SystemExit('Drive is not mounted. Run the drive.mount cell above.')

shared = Path('/content/drive/Shareddrives')
if shared.is_dir():
    print(f"Shared drives:    {[p.name for p in shared.iterdir()][:5]}")

print(f"Video folder:     {VIDEOS}")
if not VIDEOS.is_dir():
    print('\nNOT FOUND. What is next to it:')
    parent = VIDEOS.parent
    if parent.is_dir():
        for p in sorted(parent.iterdir())[:30]:
            print(f"   {'d' if p.is_dir() else '-'} {p.name}")
    else:
        print(f'   {parent} does not exist either')
    raise SystemExit(
        'Set VIDEOS to the real folder. A "Shared with me" folder needs a '
        'shortcut added to My Drive before it has a path.')

found = sorted(VIDEOS.glob(VIDEO_GLOB))
print(f"Matching {VIDEO_GLOB}: {len(found)} file(s)")
if not found:
    others = sorted({p.suffix for p in VIDEOS.iterdir() if p.is_file()})
    print(f"  extensions present in that folder: {others}")
    raise SystemExit(f'No {VIDEO_GLOB} in {VIDEOS}. Adjust VIDEO_GLOB.')
for p in found[:10]:
    print(f"   {p.name}  {p.stat().st_size/1e6:.0f} MB")
if len(found) > 10:
    print(f"   ... and {len(found)-10} more")

In [ ]:
# Open every clip, decode a frame, and compare against what is already exported.
# Frame counts come from the same OpenCV call idtracker.ai uses, so a clip that
# reads as 0 frames here would also be rejected there.
!cd {TOOLS} && python check_videos.py \
    --videos {VIDEOS}/{VIDEO_GLOB} \
    --contours {CONTOURS} \
    --fps 10

In [ ]:
# Room for the local cache. Only one clip is cached at a time, but Colab's disk
# is shared with the dataset, the wheel build and the model checkpoints.
import shutil as _shutil

total, used, free = _shutil.disk_usage('/content')
biggest = max((p.stat().st_size for p in found), default=0) / 1e9
print(f"local disk: {free/1e9:.0f} GB free of {total/1e9:.0f} GB")
print(f"largest clip: {biggest:.1f} GB")
if free / 1e9 < biggest * 3:
    print("\nTight. Either drop --local-cache from the export (slower, reads"
          "\nstraight off Drive) or free space up.")
else:
    print("Enough for --local-cache.")

## 3. Install Detectron2There is no universal Detectron2 wheel — it compiles against the installedtorch/CUDA pair, which takes about ten minutes. The build is cached to Drive andkeyed by Python and torch version, so later sessions reuse it. If Colab updatesits runtime the key changes and it rebuilds once.

In [ ]:
import subprocess, sys, torch

key = f"py{sys.version_info.major}{sys.version_info.minor}-torch{torch.__version__}"
cache = WHEELS / key
cache.mkdir(parents=True, exist_ok=True)
wheels = list(cache.glob('detectron2*.whl'))
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}, cache key {key}")

def run(cmd):
    print('$', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

try:
    import detectron2
    print('detectron2 already importable:', detectron2.__version__)
except ImportError:
    if wheels:
        print(f'Installing cached wheel {wheels[0].name}')
        run([sys.executable, '-m', 'pip', 'install', '-q', str(wheels[0])])
    else:
        print('Building detectron2 (~10 min) and caching the wheel to Drive')
        rc = run([sys.executable, '-m', 'pip', 'wheel', '--no-deps', '-q',
                  '--wheel-dir', str(cache),
                  'git+https://github.com/facebookresearch/detectron2.git'])
        wheels = list(cache.glob('detectron2*.whl'))
        if rc or not wheels:
            raise SystemExit('detectron2 build failed; see the log above')
        run([sys.executable, '-m', 'pip', 'install', '-q', str(wheels[0])])

In [ ]:
# Verify before spending GPU time on a broken install.
import torch, detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg

print('detectron2', detectron2.__version__)
print('torch      ', torch.__version__)
print('CUDA avail ', torch.cuda.is_available())
print('device     ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY')
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'))
print('model zoo config loads OK')
assert torch.cuda.is_available(), 'No CUDA. Switch the runtime to GPU and rerun.'

## 4. Check the dataset before trainingRenders a few training images with their polygons drawn on. Look at them. Anannotation that is offset, missing an animal, or attached to the wrong classcosts a whole training run to discover afterwards.

In [ ]:
!cd {TOOLS} && python train_detectron2.py \
    --dataset {DATASET} --output {MODEL} --check-dataset

In [ ]:
from IPython.display import Image, display
samples = sorted((MODEL / 'dataset_check').glob('*.png'))[:4]
print(f'{len(samples)} sample(s)')
for s in samples:
    display(Image(filename=str(s), width=600))

## 5. TrainAbout 40 epochs over ~540 images at batch 2 is roughly 10 800 iterations, whichtakes on the order of an hour on an L4. Checkpoints land in `MODEL` every fifthof the run, so a disconnect does not cost everything — rerun with `--resume`.

In [ ]:
!cd {TOOLS} && python train_detectron2.py \
    --dataset {DATASET} \
    --output {MODEL} \
    --epochs {EPOCHS} \
    --batch-size 2 \
    --min-size-test 640

In [ ]:
# If the session dropped mid-training, rerun this instead of the cell above.
# !cd {TOOLS} && python train_detectron2.py --dataset {DATASET} --output {MODEL} --epochs {EPOCHS} --resume

In [ ]:
import json
meta = json.loads((MODEL / 'training_metadata.json').read_text())
print(f"classes: {meta['class_names']}   images: {meta['train_images']} train / {meta['val_images']} val")
print(f"enhancement recorded: {meta['enhancement']}")
for task, metrics in meta.get('validation', {}).items():
    print(f"\n{task}:")
    for k, v in metrics.items():
        print(f"  {k:12s} {v:.3f}")
print('''
segm/AP is the one that matters for contours. Read it against how the
validation set was split: if train and val share source clips, this number is
optimistic. labelme_to_coco.py splits by video by default for that reason.''')

## 6. Export contours — the long stageResumable by design. Each clip writes `contours/<clip>.h5` and reruns skip filesthat already exist, so if the session dies you reconnect and run this cellagain. Videos are copied to local disk first because decoding straight off theDrive mount is much slower than the copy.Run this cell as many times as it takes.

In [ ]:
!cd {TOOLS} && python detectron2_export_contours.py \
    --videos {VIDEOS}/{VIDEO_GLOB} \
    --weights {MODEL}/model_final.pth \
    --output-dir {CONTOURS} \
    --local-cache {CACHE} \
    --max-instances {N_ANIMALS} \
    --score-threshold {SCORE_THRESHOLD} \
    --on-overlap {ON_OVERLAP} \
    --progress-every 1000

In [ ]:
# How far along is the batch, and how much longer at the observed rate?
import json

clips = sorted(VIDEOS.glob(VIDEO_GLOB))
done  = sorted(CONTOURS.glob('*.h5'))
print(f'{len(done)}/{len(clips)} clips exported')

log_path = CONTOURS / 'export_log.json'
if log_path.is_file():
    log = json.loads(log_path.read_text())
    frames = sum(e['frames'] for e in log)
    seconds = sum(e['seconds'] for e in log)
    empty = sum(e['empty_frames'] for e in log)
    short = sum(e['short_frames'] for e in log)
    fps = frames / seconds if seconds else 0
    print(f'{frames:,} frames at {fps:.1f} fps overall')
    print(f'{empty:,} frames with no detection ({100*empty/max(frames,1):.2f}%)')
    print(f'{short:,} frames with fewer than {N_ANIMALS} ({100*short/max(frames,1):.2f}%)')
    remaining = [c for c in clips if not (CONTOURS / f'{c.stem}.h5').exists()]
    if remaining and fps:
        import cv2
        left = sum(int(cv2.VideoCapture(str(c)).get(cv2.CAP_PROP_FRAME_COUNT))
                   for c in remaining)
        print(f'\n{len(remaining)} clip(s) left, ~{left/fps/3600:.1f} h at this rate')
    elif not remaining:
        print('\nAll clips exported.')

print('''
A high "no detection" or "fewer than expected" rate is worth looking at before
tracking: it usually means the score threshold is too strict, or the footage is
genuinely hard in places. Those frames are left empty on purpose - idtracker.ai
reconstructs them from the whole video.''')

## 7. Check the output, then track locallyThe `.h5` files are small — tens of MB each rather than the gigabytes anenhanced video would be. Download `contours/` from Drive to the machine runningidtracker.ai.**Track the same video files these contours were made from.** idtracker.aichecks the sidecar's frame count and resolution against the video and refuses amismatch, so a re-encoded or re-exported copy will be rejected.

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location(
    'ec', CODE / 'src/idtrackerai/base/animals_detection/external_contours.py')
ec = importlib.util.module_from_spec(spec); spec.loader.exec_module(ec)

import cv2
total_mb = 0
for h5 in sorted(CONTOURS.glob('*.h5')):
    c = ec.ExternalContours(h5)
    mb = h5.stat().st_size / 1e6
    total_mb += mb
    video = VIDEOS / f'{h5.stem}{Path(VIDEO_GLOB).suffix}'
    verdict = 'no matching video found'
    if video.is_file():
        cap = cv2.VideoCapture(str(video))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        verdict = 'matches video' if (n, w, h) == (c.n_frames, c.width, c.height) \
                  else f'MISMATCH: video is {n} frames {w}x{h}'
    print(f'{h5.name:28s} {c.n_frames:6d} frames  {c.width}x{c.height}  {mb:6.1f} MB  {verdict}')
    c.close()
print(f'\n{total_mb:.0f} MB total')

In [ ]:
# The .toml line each clip needs on the idtracker.ai machine.
for h5 in sorted(CONTOURS.glob('*.h5')):
    print(f'# {h5.stem}.toml')
    print(f'external_contours = "contours/{h5.name}"')
    print(f'number_of_animals = {N_ANIMALS}')
    print()
print('''Or pick the file in the Segmentation App:
Segmentation -> External contours -> Load contours.

Background subtraction and the intensity thresholds grey out, because they take
no part once contours come from the model. Everything after segmentation -
crossing detection, fragmentation, identification, gap closing - runs
unchanged.''')